# SMILES MIA — CUDA Arm (Google Drive Persistent & Auto-Resume)

Ye notebook aapki saari progress (checkpoints `latest.pt`, `epoch.pt`, loss curves, raw data, results) ko **Google Drive** pe continuously save karti hai.

> **Fayda**: Agar Colab disconnect ho jata hai ya credit/timeout khatam ho jata hai, to dobara run karne par training shuru se restart nahi hogi — balki Google Drive se last saved checkpoint utha kar aage se training resume hogi!

## Step 1 of 4 — Mount Google Drive & Set up persistent workspace

Run this cell. It mounts your Google Drive and creates the project folder at `/content/drive/MyDrive/smiles_mia_colab`. If `colab_package.zip` is not yet extracted in Drive, it will prompt you to upload it once.

In [ ]:
from pathlib import Path

print("Drive contents:")
for item in Path('/content/drive').iterdir():
    print(" -", item)

print("\nMyDrive contents:")
for item in Path('/content/drive/MyDrive').iterdir():
    print(" -", item.name)

In [ ]:
import os, sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# Google Drive me search karo smiles_mia_colab folder
target = None
for root, dirs, files in os.walk('/content/drive'):
    if 'colab_pipeline.py' in files and 'smiles_mia_colab' in root:
        target = Path(root)
        break

if target is None:
    # Fallback to direct path
    target = Path('/content/drive/MyDrive/smiles_mia_colab')

PROJECT = target
print("🎯 Folder connect ho gaya:", PROJECT)

os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
os.environ['PROJECT_ROOT'] = str(PROJECT)

print("Working Directory:", os.getcwd())
print("colab_pipeline.py found:", (PROJECT / 'colab_pipeline.py').is_file())

In [ ]:
import os, sys, shutil, zipfile
from pathlib import Path
from google.colab import drive, files

# 1. Google Drive Mount karein
drive.mount('/content/drive')

# 2. Persistent folder Google Drive pe banayein
PROJECT = Path('/content/drive/MyDrive/smiles_mia_colab')
PROJECT.mkdir(parents=True, exist_ok=True)

# 3. Check karein ki files already Drive me hain ya zip extract karni hai
if not (PROJECT / 'colab_pipeline.py').is_file():
    print('Looking for colab_package.zip...')
    zip_candidates = list(Path('/content').glob('*.zip')) + list(Path('/content/drive/MyDrive').glob('colab_package*.zip'))
    if not zip_candidates:
        print('colab_package.zip ko upload karein (one-time upload):')
        uploaded = files.upload()
        zip_candidates = [Path(name) for name in uploaded if name.lower().endswith('.zip')]

    if zip_candidates:
        target_zip = zip_candidates[0]
        print(f'Extracting {target_zip} into Google Drive: {PROJECT}...')
        with zipfile.ZipFile(target_zip) as zf:
            zf.extractall(PROJECT)
        print('Extraction complete!')
    else:
        raise FileNotFoundError('colab_package.zip not found! Upload it and re-run this cell.')
else:
    print(f'Project files already present in Google Drive at: {PROJECT}')

# 4. Working directory set karein Google Drive path par
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
os.environ['PROJECT_ROOT'] = str(PROJECT)

print('Working Directory (Google Drive):', os.getcwd())
print('colab_pipeline.py found:', (PROJECT / 'colab_pipeline.py').is_file())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Looking for colab_package.zip...
colab_package.zip ko upload karein (one-time upload):


## Step 2 of 4 — Install packages and check GPU/imports

This step verifies CUDA availability and sets up the project root.

In [ ]:
%pip -q install -r requirements.txt
import torch, importlib
assert torch.cuda.is_available(), 'GPU missing: Runtime > Change runtime type > T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
importlib.invalidate_caches()
import colab_pipeline
colab_pipeline.set_root(PROJECT)
from colab_pipeline import preflight, run_stages
print('Pipeline import: PASS')

## Step 3 of 4 — Preflight data and DP compatibility (no training)

Checks data split and Opacus backward pass. All prepared data will be saved directly into Google Drive (`/content/drive/MyDrive/smiles_mia_colab/data/`).

In [ ]:
preflight()

## Step 4 of 4 — Run experiments (Continuous Drive Checkpoint & Auto-Resume)

Har epoch ka checkpoint (`latest.pt`, `losses.csv`, `epoch5.pt`, `epoch20.pt`, `epoch60.pt`) Drive pe save hota hai. Agar runtime disconnect ho jaye to reconnect karke directly ye cell run karein — ye wahi se pick karega jaha se disconnect hua tha!

In [ ]:
# Step 4A: baseline, dataset-size sweep (E3), randomized-SMILES (E5a)
run_stages(['cuda_baseline', 'E3', 'E5a'])

In [ ]:
import os, sys, shutil
from pathlib import Path

# 1. Project directory set karo
PROJECT = Path('/content/drive/MyDrive/smiles_mia_colab')
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
os.environ['PROJECT_ROOT'] = str(PROJECT)

# 2. dp_model.py ko permanently theek karo
fixed_dp_model = '''"""Opacus-compatible causal GPT used only for the DP-SGD arm."""
import math
import torch
from torch import nn

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head=4, dropout=0.1):
        super().__init__()
        if d_model % n_head:
            raise ValueError('d_model must be divisible by n_head')
        self.n_head = n_head
        self.head_dim = d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.output_dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch, length, width = x.shape
        qkv = self.qkv(x).view(batch, length, 3, self.n_head, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q, k, v = (t.transpose(1, 2) for t in (q, k, v))
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        causal = torch.ones(length, length, dtype=torch.bool, device=x.device).triu(1)
        scores = scores.masked_fill(causal, torch.finfo(scores.dtype).min)
        weights = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = (weights @ v).transpose(1, 2).contiguous().view(batch, length, width)
        return self.output_dropout(self.proj(out))

class DPTransformerBlock(nn.Module):
    def __init__(self, d_model=128, n_head=4, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_head=n_head, dropout=dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.linear1 = nn.Linear(d_model, 4 * d_model)
        self.linear2 = nn.Linear(4 * d_model, d_model)
        self.activation_dropout = nn.Dropout(dropout)
        self.output_dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        feed_forward = self.linear1(self.norm2(x))
        feed_forward = self.activation_dropout(torch.relu(feed_forward))
        return x + self.output_dropout(self.linear2(feed_forward))

class DPCausalGPT(nn.Module):
    def __init__(self, vocab, n_layer=2, d_model=128, n_head=4, context=128, dropout=0.1):
        super().__init__()
        self.context = context
        self.emb = nn.Embedding(vocab, d_model)
        self.pos = nn.Embedding(context, d_model)
        self.blocks = nn.ModuleList([
            DPTransformerBlock(d_model=d_model, n_head=n_head, dropout=dropout)
            for _ in range(n_layer)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab, bias=False)

    def forward(self, x):
        batch, length = x.shape
        positions = torch.arange(length, device=x.device).unsqueeze(0).expand(batch, length)
        h = self.emb(x) + self.pos(positions)
        for block in self.blocks:
            h = block(h)
        return self.head(self.norm(h))
'''

target_file = PROJECT / 'src' / 'dp_model.py'
target_file.parent.mkdir(parents=True, exist_ok=True)
target_file.write_text(fixed_dp_model.strip())
print("✅ src/dp_model.py updated permanently!")

# 3. Memory se purane modules clear karo
for m in list(sys.modules.keys()):
    if 'dp_model' in m or 'colab_pipeline' in m:
        del sys.modules[m]

import colab_pipeline
colab_pipeline.set_root(PROJECT)
print("✅ Project root set to Google Drive:", colab_pipeline.ROOT)
print("✅ All set! Ab Step 4B run karo.")

### Step 4B — Advanced experiments (pretrain_ft, DP_SGD, E6)

Run after Step 4A completes. It runs pretraining + fine-tuning, Differential Privacy (DP-SGD), and model generation evaluation (E6).

In [ ]:
run_stages(['pretrain_ft', 'DP_SGD', 'E6'])

## Check Status & Manifest

Check which stages are completed, running, or if any error occurred.

In [ ]:
import json
manifest = PROJECT / 'results' / 'cuda_arm' / 'manifest.json'
print(manifest.read_text() if manifest.exists() else 'Manifest not created yet.')

## Download / Backup Results

Note: Aapke saare results aur checkpoints pehle se hi Google Drive me surakshit (`/content/drive/MyDrive/smiles_mia_colab/results`) hain! Agar aap local computer pe bhi download karna chahte hain to ye cell run karein.

In [ ]:
import shutil
from google.colab import files

results_dir = PROJECT / 'results'
if results_dir.exists():
    shutil.make_archive('/content/results', 'zip', results_dir)
    print('Downloading results.zip to local computer...')
    files.download('/content/results.zip')
else:
    print('results directory does not exist yet. Run the experiments first.')
